In [1]:
import copy
import os
import json
import statistics
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import torch
import logging

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
logging.getLogger("vllm").setLevel(logging.ERROR)

from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt
from PIL import Image



from transformers.generation import ContinuousBatchingConfig, ContinuousBatchingManager


/mnt/code/code/leosight-experiments/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== User-configurable settings =====
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
IMAGES_DIR = Path("images")
SYSTEM_PREFIX = "You are an assistant on board of a satellite who helps interpret disaster images."

# Multiple queries are run on each image.
QUERY_PROMPTS = [
    "Describe this image in detail. Include scene elements, objects, text, and spatial relationships.",
    "List the top 10 visible objects and where they are located in the image.",
    "What potential hazards or risks can you infer from this image?",
    "Summarize the scene in 3 concise bullet points.",
]

# Optional image filtering (set to empty tuple to allow all)
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".webp")

# Generation settings
MAX_TOKENS = 128
TEMPERATURE = 0.0

# Profiling controls
WARMUP_RUNS = 1
PROFILE_RUNS = 3

# Max number of distinct queries to send in one batched call.
# In continuous batching mode, queries are grouped into chunks of this size.
BATCH_SIZE = 4


# Visual token compression knobs (official processor pixel budget)
# Lower max_pixels -> fewer visual tokens -> faster but potentially less detail.
MAX_PIXELS = 512 * 512
MIN_PIXELS = 256 * 256

print('Configuration loaded.')

Configuration loaded.


In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    enable_prefix_caching: bool
    use_continuous_batching: bool
    use_visual_token_compression: bool


EXPERIMENTS = [
    ExperimentConfig(
        name="baseline",
        enable_prefix_caching=False,
        use_continuous_batching=False,
        use_visual_token_compression=False,
    ),
    ExperimentConfig(
        name="continuous_batching_only",
        enable_prefix_caching=False,
        use_continuous_batching=True,
        use_visual_token_compression=False,
    ),
    ExperimentConfig(
        name="prefix_caching_only",
        enable_prefix_caching=True,
        use_continuous_batching=False,
        use_visual_token_compression=False,
    ),
    ExperimentConfig(
        name="visual_token_compression_only",
        enable_prefix_caching=False,
        use_continuous_batching=False,
        use_visual_token_compression=True,
    ),
    ExperimentConfig(
        name="cb_prefix_vtc",
        enable_prefix_caching=True,
        use_continuous_batching=True,
        use_visual_token_compression=True,
    ),
]


def get_image_paths(images_dir: Path):
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir.resolve()}")

    paths = []
    for p in sorted(images_dir.iterdir()):
        if p.is_file() and (not IMAGE_EXTENSIONS or p.suffix.lower() in IMAGE_EXTENSIONS):
            paths.append(p.resolve())

    if not paths:
        raise ValueError(f"No image files found in {images_dir.resolve()}")

    return paths




def build_llm(config: ExperimentConfig):


    mm_processor_kwargs = {}


    if config.use_visual_token_compression:
        mm_processor_kwargs["max_pixels"] = MAX_PIXELS
        mm_processor_kwargs["min_pixels"] = MIN_PIXELS


    llm = LLM(
        model=MODEL_NAME,
        # Enable prefix caching at the engine level if requested
        enable_prefix_caching=config.enable_prefix_caching,
        max_model_len = 4096,
        tensor_parallel_size=2, 
        mm_processor_kwargs=mm_processor_kwargs,
    )
    return {"model": llm}


def run_once(llm_bundle, config: ExperimentConfig, image_path: Path, query_batch: list[str]):
    llm = llm_bundle["model"]

    if not query_batch:
        raise ValueError("query_batch cannot be empty")

    with Image.open(image_path) as img:
        base_image = img.convert("RGB")

    sampling_params = SamplingParams(
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE if TEMPERATURE > 0 else 0.0,
    )

    # Build vLLM inputs: each prompt is a dict with "prompt" text + "multi_modal_data"
    prompts = []
    for query_text in query_batch:
        messages = [
            {"role": "system", "content": SYSTEM_PREFIX},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": "placeholder"}},
                    {"type": "text", "text": query_text},
                ],
            },
        ]
        # Use the model's chat template via vLLM's tokenizer
        prompt_text = llm.get_tokenizer().apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append({
            "prompt": prompt_text,
            "multi_modal_data": {"image": base_image.copy()},
        })

    t0 = time.perf_counter()
    outputs = llm.generate(prompts, sampling_params=sampling_params)
    t1 = time.perf_counter()

    per_query = []
    total_output_tokens = 0

    for i, request_output in enumerate(outputs):
        # vLLM returns RequestOutput; use the first (and only) CompletionOutput
        completion = request_output.outputs[0]

        prompt_tokens = len(request_output.prompt_token_ids)
        output_tokens = len(completion.token_ids)
        total_output_tokens += output_tokens

        per_query.append({
            "query_text": query_batch[i],
            "prompt_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "sample_text": completion.text.strip()[:240],
        })

    elapsed_s = t1 - t0
    tok_per_s = (total_output_tokens / elapsed_s) if elapsed_s > 0 else 0.0
    prompt_lens = [x["prompt_tokens"] for x in per_query]

    return {
        "elapsed_s": elapsed_s,
        "requests_in_call": len(query_batch),
        "avg_prompt_tokens": float(statistics.mean(prompt_lens)) if prompt_lens else 0.0,
        "avg_output_tokens": float(statistics.mean([x["output_tokens"] for x in per_query])) if per_query else 0.0,
        "total_output_tokens": total_output_tokens,
        "throughput_output_tok_per_s": tok_per_s,
        "avg_first_token_latency_s": None,  # Not available in offline mode; use AsyncEngine for TTFT
        "items": per_query,
    }


print("Helpers ready.")

Helpers ready.


In [4]:
image_paths = get_image_paths(IMAGES_DIR)
query_prompts = QUERY_PROMPTS
query_to_idx = {q: i for i, q in enumerate(query_prompts, start=1)}

print(f"Found {len(image_paths)} images in {IMAGES_DIR.resolve()}")
for p in image_paths:
    print(f"  - {p.name}")

print(f"\nUsing {len(query_prompts)} queries per image")
for i, q in enumerate(query_prompts, start=1):
    print(f"  Q{i}: {q}")

all_raw_runs = []
summary_rows = []

for exp in EXPERIMENTS:
    print(f'\n=== Running experiment: {exp.name} ===')

    llm_bundle = build_llm(exp)

    batch_size = BATCH_SIZE if exp.use_continuous_batching else 1
    print(f"Batch size for this experiment: {batch_size}")

    # Warmup on first image and first query.
    for i in range(WARMUP_RUNS):
        _ = run_once(llm_bundle, exp, image_paths[0], [query_prompts[0]])
        print(f'  warmup {i + 1}/{WARMUP_RUNS} complete')


    run_metrics = []
    for image_path in image_paths:
        for i in range(PROFILE_RUNS):
            batch_metrics_this_run = []

            for start in range(0, len(query_prompts), batch_size):
                query_batch = query_prompts[start : start + batch_size]
                metrics = run_once(llm_bundle, exp, image_path, query_batch)
                batch_metrics_this_run.append(metrics)

                for item in metrics["items"]:
                    all_raw_runs.append(
                        {
                            "experiment": exp.name,
                            "image_name": image_path.name,
                            "query_idx": query_to_idx[item["query_text"]],
                            "query_text": item["query_text"],
                            "run_idx": i + 1,
                            "elapsed_s": metrics["elapsed_s"],
                            "requests_in_call": metrics["requests_in_call"],
                            "avg_prompt_tokens": float(item["prompt_tokens"]),
                            "avg_output_tokens": float(item["output_tokens"]),
                            "total_output_tokens": metrics["total_output_tokens"],
                            "throughput_output_tok_per_s": metrics["throughput_output_tok_per_s"],
                            "avg_first_token_latency_s": metrics["avg_first_token_latency_s"],
                            "sample_text": item["sample_text"],
                        }
                    )

                print(
                    f"  {image_path.name} | Q{query_to_idx[query_batch[0]]}-Q{query_to_idx[query_batch[-1]]} "
                    f"| run {i + 1}/{PROFILE_RUNS}: {metrics['elapsed_s']:.3f}s"
                )

        # Aggregate all batches for this (image, run) into one comparable metric
        total_elapsed = sum(m["elapsed_s"] for m in batch_metrics_this_run)
        total_output_tokens = sum(m["total_output_tokens"] for m in batch_metrics_this_run)
        all_items = [item for m in batch_metrics_this_run for item in m["items"]]
        ftl_vals_run = [m["avg_first_token_latency_s"] for m in batch_metrics_this_run if m["avg_first_token_latency_s"] is not None]

        run_metrics.append({
            "elapsed_s": total_elapsed,
            "total_output_tokens": total_output_tokens,
            "requests_in_call": len(all_items),
            "throughput_output_tok_per_s": total_output_tokens / total_elapsed if total_elapsed > 0 else 0.0,
            "avg_first_token_latency_s": statistics.mean(ftl_vals_run) if ftl_vals_run else None,
        })

        print(
            f"  {image_path.name} | run {i + 1}/{PROFILE_RUNS} total: {total_elapsed:.3f}s "
            f"({total_output_tokens} tokens, {run_metrics[-1]['throughput_output_tok_per_s']:.1f} tok/s)"
        )

    # Aggregate across all calls for this experiment.
    elapsed_vals = [r['elapsed_s'] for r in run_metrics]
    tps_vals = [r['throughput_output_tok_per_s'] for r in run_metrics]
    ftl_vals = [r['avg_first_token_latency_s'] for r in run_metrics if r['avg_first_token_latency_s'] is not None]

    summary_rows.append({
        "experiment": exp.name,
        "use_continuous_batching": exp.use_continuous_batching,
        "enable_prefix_caching": exp.enable_prefix_caching,
        "use_visual_token_compression": exp.use_visual_token_compression,
        "images_count": len(image_paths),
        "queries_per_image": len(query_prompts),
        "runs_per_query": PROFILE_RUNS,
        "total_samples": len(run_metrics),
        "mean_elapsed_s": statistics.mean(elapsed_vals),
        "stdev_elapsed_s": statistics.pstdev(elapsed_vals) if len(elapsed_vals) > 1 else 0.0,
        "mean_output_tok_per_s": statistics.mean(tps_vals),
        "mean_first_token_latency_s": statistics.mean(ftl_vals) if ftl_vals else None,
        "requests_per_call": run_metrics[0]['requests_in_call'] if run_metrics else None,
    })

    # Release engine before next config to reduce memory pressure.
    del llm_bundle

results_df = pd.DataFrame(summary_rows).sort_values('mean_elapsed_s')
raw_runs_df = pd.DataFrame(all_raw_runs)

display(results_df)
print('\nRaw run-level metrics:')
display(raw_runs_df)

Found 10 images in /mnt/code/code/leosight-experiments/images
  - guatemala-volcano_00000003_post_disaster.png
  - guatemala-volcano_00000003_pre_disaster.png
  - guatemala-volcano_00000005_post_disaster.png
  - guatemala-volcano_00000005_pre_disaster.png
  - guatemala-volcano_00000009_post_disaster.png
  - guatemala-volcano_00000009_pre_disaster.png
  - guatemala-volcano_00000011_post_disaster.png
  - guatemala-volcano_00000011_pre_disaster.png
  - guatemala-volcano_00000021_post_disaster.png
  - guatemala-volcano_00000021_pre_disaster.png

Using 4 queries per image
  Q1: Describe this image in detail. Include scene elements, objects, text, and spatial relationships.
  Q2: List the top 10 visible objects and where they are located in the image.
  Q3: What potential hazards or risks can you infer from this image?
  Q4: Summarize the scene in 3 concise bullet points.

=== Running experiment: baseline ===


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


[Gloo] Rank [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 11 is connected to 
1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(Worker pid=1923934) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=1923933) (Worker pid=1923934) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=1923933) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


...

Processed prompts: 100%|██████████| 4/4 [00:00<00:00,  6.98it/s, est. speed input: 2099.46 toks/s, output: 766.11 toks/s]


  guatemala-volcano_00000021_pre_disaster.png | Q1-Q4 | run 3/3: 0.601s
  guatemala-volcano_00000021_pre_disaster.png | run 3/3 total: 0.601s (439 tokens, 730.4 tok/s)


,experiment,use_continuous_batching,enable_prefix_caching,use_visual_token_compression,images_count,queries_per_image,runs_per_query,total_samples,mean_elapsed_s,stdev_elapsed_s,mean_output_tok_per_s,mean_first_token_latency_s,requests_per_call
4,cb_prefix_vtc,True,True,True,10,4,3,10,0.598527,0.009854,745.989931,None,4
1,continuous_batching_only,True,False,False,10,4,3,10,0.897492,0.005219,501.418140,None,4
2,prefix_caching_only,True,True,False,10,4,3,10,1.958285,0.050464,230.520906,None,4
3,visual_token_compression_only,True,False,True,10,4,3,10,1.958389,0.043775,228.215128,None,4
0,baseline,True,False,False,10,4,3,10,2.251235,0.084540,202.004746,None,4



Raw run-level metrics:


,experiment,image_name,query_idx,query_text,run_idx,elapsed_s,requests_in_call,avg_prompt_tokens,avg_output_tokens,total_output_tokens,throughput_output_tok_per_s,avg_first_token_latency_s,sample_text
0,baseline,guatemala-volcano_00000003_post_disaster.png,1,Describe this image in detail. Include scene e...,1,0.623034,1,1072.0,128.0,128,205.446108,None,"This is an aerial, high-angle satellite image ..."
1,baseline,guatemala-volcano_00000003_post_disaster.png,2,List the top 10 visible objects and where they...,1,0.617895,1,1071.0,128.0,128,207.155081,None,- A house located in the upper left portion of...
2,baseline,guatemala-volcano_00000003_post_disaster.png,3,What potential hazards or risks can you infer ...,1,0.616912,1,1066.0,128.0,128,207.485068,None,Based on the visual evidence in this satellite...
3,baseline,guatemala-volcano_00000003_post_disaster.png,4,Summarize the scene in 3 concise bullet points.,1,0.472478,1,1066.0,93.0,93,196.834734,None,- A satellite image shows a landscape with a d...
4,baseline,guatemala-volcano_00000003_post_disaster.png,1,Describe this image in detail. Include scene e...,2,0.642769,1,1072.0,128.0,128,199.138467,None,"This is an aerial, high-angle satellite image ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,cb_prefix_vtc,guatemala-volcano_00000021_pre_disaster.png,4,Summarize the scene in 3 concise bullet points.,2,0.593862,4,298.0,55.0,439,739.228859,None,- A large agricultural complex with a greenhou...
596,cb_prefix_vtc,guatemala-volcano_00000021_pre_disaster.png,1,Describe this image in detail. Include scene e...,3,0.601051,4,304.0,128.0,439,730.387547,None,This is an aerial photograph of a rural landsc...
597,cb_prefix_vtc,guatemala-volcano_00000021_pre_disaster.png,2,List the top 10 visible objects and where they...,3,0.601051,4,303.0,128.0,439,730.387547,None,- A large agricultural area with a greenhouse ...
598,cb_prefix_vtc,guatemala-volcano_00000021_pre_disaster.png,3,What potential hazards or risks can you infer ...,3,0.601051,4,298.0,128.0,439,730.387547,None,"Based on the visual information in the image, ..."


In [5]:
# Save outputs for later analysis

safe_model_name = MODEL_NAME.replace("/", "_")

results_df.to_csv(f'{safe_model_name}_qwen3vl_vllm_profile_summary.csv', index=False)
raw_runs_df.to_csv(f'{safe_model_name}_qwen3vl_vllm_profile_raw_runs.csv', index=False)

with open(f'{safe_model_name}_qwen3vl_vllm_profile_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_rows, f, indent=2)

with open(f'{safe_model_name}_qwen3vl_vllm_profile_raw_runs.json', 'w', encoding='utf-8') as f:
    json.dump(all_raw_runs, f, indent=2)

print('Saved:')
print('  ' + f'{safe_model_name}_qwen3vl_vllm_profile_summary.csv')
print('  ' + f'{safe_model_name}_qwen3vl_vllm_profile_raw_runs.csv')
print('  ' + f'{safe_model_name}_qwen3vl_vllm_profile_summary.json')
print('  ' + f'{safe_model_name}_qwen3vl_vllm_profile_raw_runs.json')

Saved:
  Qwen_Qwen3-VL-2B-Instruct_qwen3vl_vllm_profile_summary.csv
  Qwen_Qwen3-VL-2B-Instruct_qwen3vl_vllm_profile_raw_runs.csv
  Qwen_Qwen3-VL-2B-Instruct_qwen3vl_vllm_profile_summary.json
  Qwen_Qwen3-VL-2B-Instruct_qwen3vl_vllm_profile_raw_runs.json
